# Tutorial 06 — Multilingual and low-resource modelling; building data for African languages

[Lecture notes](../lectures/06_multilingual.md) · [Slides](../slides/06_multilingual.pptx) · [Exercise sheet](../exercises/06_multilingual.md)

**40 minutes, CPU, NumPy only.** You will see one overall score hide a failing language, put an interval on a paired comparison, weigh languages for sampling, and measure how far two annotators agree beyond chance. Every table here is invented or simulated. The language names are labels on synthetic numbers and say nothing about any model or dataset.

| Minutes | Step |
|---|---|
| 0–5 | One overall score, two different systems |
| 5–15 | Paired and clustered bootstrap; macro-F1 |
| 15–20 | Language sampling weights |
| 20–30 | Cohen's kappa and three annotators |
| 30–40 | Your turn: two functions with tests |

In [1]:
import sys
import numpy as np
rng = np.random.default_rng(42)
print("Python", sys.version.split()[0], "NumPy", np.__version__, "seed", 42)

Python 3.13.1 NumPy 2.1.3 seed 42


## 1. One overall score, two different systems

A test set has 100 English items and 10 Hausa items. System A gets 90 and 3 right. System B gets 86 and 7 right. Predict the overall accuracy of each before you run the cell.

In [2]:
language=np.array(["English"]*100+["Hausa"]*10)
correct_a=np.array([1]*90+[0]*10+[1]*3+[0]*7)
correct_b=np.array([1]*86+[0]*14+[1]*7+[0]*3)
def report(c):
    slices={l:{"n":int((language==l).sum()),"accuracy":float(c[language==l].mean())} for l in np.unique(language)}
    return {"overall":float(c.mean()),"equal_language":float(np.mean([x["accuracy"] for x in slices.values()])),"slices":slices}
print("A",report(correct_a)); print("B",report(correct_b))
assert np.isclose(correct_a.mean(),correct_b.mean())

A {'overall': 0.8454545454545455, 'equal_language': 0.6, 'slices': {np.str_('English'): {'n': 100, 'accuracy': 0.9}, np.str_('Hausa'): {'n': 10, 'accuracy': 0.3}}}
B {'overall': 0.8454545454545455, 'equal_language': 0.78, 'slices': {np.str_('English'): {'n': 100, 'accuracy': 0.86}, np.str_('Hausa'): {'n': 10, 'accuracy': 0.7}}}


Both systems score 93/110 = 0.845 overall. Averaged over languages, A scores 0.60 and B scores 0.78, because A gets 3 of 10 Hausa items right and B gets 7. Which system would you ship to Hausa speakers, and which number would have told you?

## 2. Paired and clustered bootstrap; macro-F1

Both systems answered the same 110 items, so we resample item indices once and apply them to both. That is the paired bootstrap. The second half of the cell supposes that every five consecutive items were written by one author, and resamples the 22 authors.

In [3]:
idx=rng.integers(0,len(language),size=(2000,len(language)))
diff=(correct_b[idx]-correct_a[idx]).mean(axis=1)
print("B-A observed",float(np.mean(correct_b-correct_a)),"95% percentile interval",np.quantile(diff,[.025,.975]))
def macro_f1(y,p,classes):
    scores=[]
    for c in classes:
        tp=np.sum((y==c)&(p==c)); fp=np.sum((y!=c)&(p==c)); fn=np.sum((y==c)&(p!=c))
        scores.append(2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.)
    return float(np.mean(scores))
y=np.array([0,0,0,1,1,2]); p=np.array([0,0,0,0,1,0])
print("accuracy",np.mean(y==p),"macro F1 (absent-class zero)",macro_f1(y,p,[0,1,2]))

hausa = np.where(language == "Hausa")[0]
idx_h = rng.integers(0, len(hausa), size=(2000, len(hausa)))
diff_h = (correct_b[hausa][idx_h] - correct_a[hausa][idx_h]).mean(axis=1)
print("Hausa only: B-A observed", float((correct_b[hausa] - correct_a[hausa]).mean()), "95% interval", np.quantile(diff_h, [.025, .975]))

author = np.arange(len(language)) // 5                       # suppose 22 authors wrote 5 items each
gain = np.array([(correct_b - correct_a)[author == a].sum() for a in range(22)])
idx_c = rng.integers(0, 22, size=(2000, 22))
diff_c = gain[idx_c].sum(axis=1) / len(language)
print("all items, resampling authors: 95% interval", np.quantile(diff_c, [.025, .975]))
assert np.quantile(diff_c, .975) - np.quantile(diff_c, .025) > np.quantile(diff, .975) - np.quantile(diff, .025)

B-A observed 0.0 95% percentile interval [-0.04545455  0.05454545]
accuracy 0.6666666666666666 macro F1 (absent-class zero) 0.47222222222222215
Hausa only: B-A observed 0.4 95% interval [0.1 0.7]
all items, resampling authors: 95% interval [-0.09090909  0.07272727]


Over all items the difference is 0.0 with an interval of about ±0.05. On the ten Hausa items it is 0.4, and the interval runs from 0.1 to 0.7: ten items cannot say more than that. Resampling authors widens the all-items interval from [−0.045, 0.055] to [−0.091, 0.073], because items by one author rise and fall together. The macro-F1 line scores a six-item example at accuracy 0.667 and macro-F1 0.472: class 2 is never predicted and scores zero.

## 3. Language sampling weights

Three languages have 10,000, 1,000 and 100 training examples. Sample language $l$ with probability $q_l=n_l^\alpha/\sum_j n_j^\alpha$. The last column is how many times each example is seen in 10,000 draws.

In [4]:
sizes=np.array([10000,1000,100])
for alpha in [1.,.5,0.]:
    q=sizes**alpha; q=q/q.sum()
    print("alpha",alpha,"sampling",q,"expected repeats in 10000 draws",10000*q/sizes)
    assert np.isclose(q.sum(),1)

alpha 1.0 sampling [0.9009009  0.09009009 0.00900901] expected repeats in 10000 draws [0.9009009 0.9009009 0.9009009]
alpha 0.5 sampling [0.70610111 0.22328878 0.07061011] expected repeats in 10000 draws [0.70610111 2.23288777 7.06101112]
alpha 0.0 sampling [0.33333333 0.33333333 0.33333333] expected repeats in 10000 draws [ 0.33333333  3.33333333 33.33333333]


At $\alpha=1$ every example is seen 0.9 times. At $\alpha=0$ each language gets a third of the draws, and each of the 100 examples of the smallest language is seen 33 times. Smoothing gives the small language more turns. It gives it no new sentences. What would you watch on the development set to catch memorisation?

## 4. Cohen's kappa and three annotators

Two annotators label 100 tweets as positive or negative. The table counts each pair of labels. Observed agreement $p_o$ is the diagonal share. Expected agreement $p_e$ is what two annotators with those label rates would reach by chance. $\kappa=(p_o-p_e)/(1-p_e)$.

The second table has the same form for a rare class, as in hate-speech annotation. Predict which table has the higher kappa.

In [5]:
def kappa_from_table(table):
    n = table.sum()
    p_o = np.trace(table) / n
    p_e = (table.sum(axis=1) / n) @ (table.sum(axis=0) / n)
    return p_o, p_e, (p_o - p_e) / (1 - p_e)

balanced = np.array([[40, 20], [10, 30]])       # rows: annotator 1 (positive, negative); columns: annotator 2
rare = np.array([[2, 4], [4, 90]])              # rows and columns: (hateful, not hateful)
for name, table in [("balanced", balanced), ("rare class", rare)]:
    p_o, p_e, kappa = kappa_from_table(table)
    print(f"{name:11s} observed {p_o:.3f}  expected {p_e:.3f}  kappa {kappa:.3f}")
assert np.isclose(kappa_from_table(balanced)[2], 0.4)

balanced    observed 0.700  expected 0.500  kappa 0.400
rare class  observed 0.920  expected 0.887  kappa 0.291


The rare-class table has 92% raw agreement and a kappa of 0.29. The balanced table has 70% agreement and a kappa of 0.40. Two annotators who both say "not hateful" almost every time agree often by chance, and kappa removes that part.

Now simulate a three-class sentiment task. 600 tweets have a true label. 30% are code-switched. Each of three annotators is wrong 10% of the time on tweets in one language and 35% of the time on code-switched tweets. The error rates are invented.

In [6]:
n_items = 600
truth = rng.choice(3, size=n_items, p=[.3, .4, .3])            # 0 negative, 1 neutral, 2 positive
mixed = rng.random(n_items) < .3                               # code-switched tweets

def annotate(err_plain, err_mixed):
    wrong = rng.random(n_items) < np.where(mixed, err_mixed, err_plain)
    return np.where(wrong, (truth + rng.integers(1, 3, size=n_items)) % 3, truth)

ann = np.stack([annotate(.10, .35) for _ in range(3)])

def count_table(a, b, k=3):
    table = np.zeros((k, k), int); np.add.at(table, (a, b), 1); return table

for name, keep in [("all tweets", np.ones(n_items, bool)), ("one language", ~mixed), ("code-switched", mixed)]:
    p_o, p_e, kappa = kappa_from_table(count_table(ann[0][keep], ann[1][keep]))
    print(f"{name:14s} n {keep.sum():3d}  observed {p_o:.3f}  expected {p_e:.3f}  kappa {kappa:.3f}")

votes = np.stack([(ann == c).sum(axis=0) for c in range(3)])
majority, top = votes.argmax(axis=0), votes.max(axis=0)
print(f"all three agree {(top == 3).mean():.3f}   two of three {(top == 2).mean():.3f}   no majority {(top == 1).mean():.3f}")
print(f"one annotator matches the true label {(ann[0] == truth).mean():.3f}   majority label does {(majority == truth)[top > 1].mean():.3f}")
assert kappa_from_table(count_table(ann[0][mixed], ann[1][mixed]))[2] < kappa_from_table(count_table(ann[0][~mixed], ann[1][~mixed]))[2]

all tweets     n 600  observed 0.718  expected 0.335  kappa 0.577
one language   n 424  observed 0.825  expected 0.335  kappa 0.738
code-switched  n 176  observed 0.460  expected 0.334  kappa 0.189
all three agree 0.607   two of three 0.360   no majority 0.033
one annotator matches the true label 0.842   majority label does 0.952


Kappa is 0.58 over all tweets, 0.74 on tweets in one language and 0.19 on code-switched tweets. One overall kappa hides where the guidelines fail, as one overall accuracy hid Hausa in section 1. A majority of three is right on 95% of the tweets that have a majority, against 84% for one annotator. The 3% with no majority go to an adjudicator.

## 5. Your turn

Replace each `raise` with your code, then run the test cell under it. Do not edit the tests.

**A. Cohen's kappa from two label arrays.** `a` and `b` hold one label per item from two annotators. Any number of classes. Rule for the edge case: if expected agreement is 1, return 1.0.

In [ ]:
# YOUR TURN
def cohen_kappa(a, b):
    """Cohen's kappa for two equal-length label arrays. Returns 1.0 when expected agreement is 1."""
    raise NotImplementedError("your turn")

In [8]:
a2 = np.array([1] * 60 + [0] * 40); b2 = np.array([1] * 40 + [0] * 20 + [1] * 10 + [0] * 30)   # the balanced table
assert np.isclose(cohen_kappa(a2, b2), 0.4)
assert np.isclose(cohen_kappa(ann[0], ann[1]), kappa_from_table(count_table(ann[0], ann[1]))[2])
assert np.isclose(cohen_kappa(np.array(["pos", "neg", "neu"]), np.array(["pos", "neg", "neu"])), 1.0)
assert cohen_kappa(np.zeros(5, int), np.zeros(5, int)) == 1.0
print("kappa for annotators 1 and 3:", round(float(cohen_kappa(ann[0], ann[2])), 3))
print("A passed")

kappa for annotators 1 and 3: 0.593
A passed


**B. Macro-F1 per language.** Task 12 ranks systems on macro-F1 computed per language and then averaged. Return a dictionary from language to macro-F1. Rule for absent classes: within a language, leave out a class that has no gold items and no predicted items. A class with gold or predicted items and no correct prediction scores 0.

In [ ]:
# YOUR TURN
def macro_f1_by_language(y, p, lang, classes):
    """{language: macro-F1}. In each language, skip classes with no gold and no predicted items."""
    raise NotImplementedError("your turn")

In [10]:
lang_b = np.array(["ha"] * 6 + ["yo"] * 4)
y_b = np.array([0, 0, 0, 1, 1, 2, 0, 0, 1, 1])
p_b = np.array([0, 0, 0, 0, 1, 0, 0, 1, 1, 1])
scores = macro_f1_by_language(y_b, p_b, lang_b, [0, 1, 2])
assert np.isclose(scores["ha"], (6 / 8 + 2 / 3 + 0) / 3)       # class 2 has a gold item and is never predicted: 0
assert np.isclose(scores["yo"], (2 / 3 + 4 / 5) / 2)           # class 2 is absent from "yo": left out
print({k: round(float(v), 3) for k, v in scores.items()}, "mean over languages", round(float(np.mean(list(scores.values()))), 3))
print("B passed")

{'ha': 0.472, 'yo': 0.733} mean over languages 0.603
B passed


## Discussion

1. Why might uniform language sampling hurt a very small language?
2. Can translated evaluation data tell you how a system does on naturally occurring code-switching?
3. Why is macro-F1 useful and still not enough for a multilingual system?
4. Kappa on your pilot batch is 0.19 for code-switched tweets. What do you change before annotating the next 5,000?

## From the toy to real data

`course/extensions/afrisenti_baseline.py` fits a character TF-IDF classifier on TSV files that you export yourself from a dataset you are permitted to use. It does not download or redistribute tweets. Read the [extension guide](../extensions/README.md) first.

Task 12 teams: compute macro-F1 per language on your development split with your function from part B, and report it beside the average.

Submit this notebook with both tests passing and a 150–250-word note: which interval from section 2 you would report for the Hausa comparison and why, and one thing the simulated annotators leave out.